# Computing Tempered Energy 

We aim to implement a parallel tempering scheme for annealed MCMC as per Du et al.'s [2024 paper](https://arxiv.org/pdf/2302.11552) on compositional generation with energy-based diffusion. In order to implement parallel tempering into the sampling regime of annealed MCMC, we aim to evaluate the difference between true tempered energy and our approximations.

Training follows standard DDPM noise-matching objective

### Setup

In [1]:
# from src.eval import plot_temperature_triptych

In [2]:
# k = 2.0
# n_replicas = 3
# n_samples = 4
# swap_algorithm = {
# 	"p_ratio" : "p",
# 	"even": [90, 70, 50, 40],
# 	"odd":  [89, 69, 49, 39],
# 	"debug" : False
# }

# Composed Dataset

In [3]:
# _ = plot_temperature_triptych(
# 	dataset_name="composed",
# 	k=0.5,
# 	n_samples=n_samples,
# 	n_replicas=n_replicas,
# 	replica_swaps=True,
# 	swap_algorithm=swap_algorithm
# )

In [4]:
# _ = plot_temperature_triptych(
# 	dataset_name="composed",
# 	k=k,
# 	n_samples=n_samples,
# 	n_replicas=n_replicas,
# 	replica_swaps=False,
# 	swap_algorithm=swap_algorithm
# )

# MNIST Dataset

In [5]:
# _ = plot_temperature_triptych(
# 	dataset_name="mnist",
# 	k=0.5,
# 	n_samples=n_samples,
# 	n_replicas=n_replicas,
# 	replica_swaps=False,
# 	swap_algorithm=swap_algorithm
# )

In [6]:
# _ = plot_temperature_triptych(
# 	dataset_name="mnist",
# 	k=2.0,
# 	n_samples=n_samples,
# 	n_replicas=n_replicas,
# 	replica_swaps=False,
# 	swap_algorithm=swap_algorithm
# )

# Images

In [ ]:
import torch
from diffusers import StableDiffusion3Pipeline

: 

In [ ]:
pipe = StableDiffusion3Pipeline.from_pretrained(
    "stabilityai/stable-diffusion-3-medium-diffusers",
    torch_dtype=torch.float16,
    cache_dir="/n/netscratch/kempner_undergrads/Everyone/zwu/parallel_toy/model_checkpoints"
)
pipe = pipe.to("cuda")

Loading pipeline components...:   0%|          | 0/9 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/517 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/219 [00:00<?, ?it/s]

In [ ]:
tsr_k = 0.75
tsr_sigma = 1.0
replica_exchange = True
swap_algorithm={
        "n_replicas": 3,
        "p_ratio": "p",
        "even_timesteps": [90.0, 70.0, 50.0, 40.0],
        "odd_timesteps": [89.0, 69.0, 49.0, 39.0],
		"debug": True
    }

generator = torch.Generator(device="cuda").manual_seed(42)

all_images = pipe(
    "hyperrealism chiaroscuro cinematic oil on canvas matte painting of professional golden hour flambient real estate photo scifi traditional japanese onsen shinto temple zen garden",
    negative_prompt="",
    num_inference_steps=28,
    guidance_scale=7.0,
    tsr_k=tsr_k,
    tsr_sigma=tsr_sigma,
    replica_exchange=replica_exchange,
    swap_algorithm=swap_algorithm,
    generator=generator,
).images

for idx in range(len(all_images)):

	image = all_images[idx]

	output_path = f"/n/netscratch/kempner_undergrads/Everyone/zwu/parallel_toy/figures/figure_k{tsr_k}_idx{idx}_{replica_exchange}.png"
	image.save(output_path)
	print(f"Saved to {output_path}")

# Other

In [ ]:
# import torch
# import matplotlib.pyplot as plt
# import numpy as np

# n_samples=2


# def apply_temperature(x, k=1.0):
#     """x in [0,1]. k>1 sharpens, k<1 flattens."""
#     x = x.clamp(1e-6, 1 - 1e-6)  # avoid log(0)
#     logits = torch.log(x / (1 - x))  # inverse sigmoid
#     logits = logits * k              # scale by temperature
#     return torch.sigmoid(logits)     # back to [0,1]

# x = load_mnist_tensor(train=False, normalize_to_minus1_1=False)

# fig, axes = plt.subplots(n_samples, len(k_range), figsize=(len(k_range) * 2, n_samples*2))

# for row in range(n_samples):
#     for col, k in enumerate(k_range):
#         img = apply_temperature(x[row, 0], k=k)
#         axes[row, col].imshow(img.numpy(), cmap="gray", vmin=0, vmax=1)
#         axes[row, col].axis("off")
#         if row == 0:
#             axes[row, col].set_title(f"k={k:.2f}")

# plt.tight_layout()
# plt.show()